# LAB3 - Bài 4

## Câu a

In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.model_selection import KFold
from sklearn import metrics

import keras
from keras.models import Sequential
from keras.layers import Dense, Input

In [2]:
# Cố định seed để kết quả huấn luyện Neural Network có thể lặp lại
keras.utils.set_random_seed(42)

In [3]:
# Đọc bộ dữ liệu Iris từ sklearn
iris = load_iris()
print("Feature names:", iris.feature_names)
print("Target names:", iris.target_names)
print("Data shape:", iris.data.shape, "- Target shape:", iris.target.shape)

Feature names: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Target names: ['setosa' 'versicolor' 'virginica']
Data shape: (150, 4) - Target shape: (150,)


Đặng Phan Duy - 6451071010

## Câu b

In [4]:
X = iris.data
Y = iris.target

# Chuẩn hóa về đoạn [0,1]: chia mỗi giá trị cho giá trị lớn nhất của cột tương ứng
X_max = X.max(axis=0)
X = X / X_max

print("Max của từng cột:", X_max)
print("X sau khi chuẩn hóa:\n", X[:5])
print("Min:", X.min(), "- Max:", X.max())

Max của từng cột: [7.9 4.4 6.9 2.5]
X sau khi chuẩn hóa:
 [[0.64556962 0.79545455 0.20289855 0.08      ]
 [0.62025316 0.68181818 0.20289855 0.08      ]
 [0.59493671 0.72727273 0.1884058  0.08      ]
 [0.58227848 0.70454545 0.2173913  0.08      ]
 [0.63291139 0.81818182 0.20289855 0.08      ]]
Min: 0.04 - Max: 1.0


Đặng Phan Duy - 6451071010

## Câu c

In [5]:
# Hàm tạo và biên dịch Neural Network (tương tự câu 1.h)
def create_model():
    model = Sequential([
        Input(shape=(X.shape[1],)),                             # 4 thuộc tính
        Dense(10, activation='relu'),                           # hidden layer 1: 10 neuron
        Dense(20, activation='relu'),                           # hidden layer 2: 20 neuron
        Dense(len(iris.target_names), activation='softmax')     # output layer: 3 class
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

create_model().summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 10)                  │              50 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 20)                  │             220 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 3)                   │              63 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 333 (1.30 KB)

 Trainable params: 333 (1.30 KB)

 Non-trainable params: 0 (0.00 B)

Đặng Phan Duy - 6451071010

## Câu d

In [6]:
# Huấn luyện mô hình bằng k-fold cross validation (k = 10)
kfold = KFold(n_splits=10, shuffle=True, random_state=42)

models = []
accuracy_per_fold = []
precision_per_fold = []
recall_per_fold = []
f1_per_fold = []
class_precision, class_recall, class_f1 = [], [], []   # độ đo của từng class trong mỗi fold

fold_no = 1
for train_index, val_index in kfold.split(X):
    model = create_model()

    X_train_fold, X_val_fold = X[train_index], X[val_index]
    y_train_fold, y_val_fold = Y[train_index], Y[val_index]

    # Huấn luyện giống câu 1.h (epochs = 100, batch_size = 1); verbose=0 để không in 100 epoch x 10 fold
    model.fit(X_train_fold, y_train_fold, epochs=100, batch_size=1, verbose=0)

    # Lưu model huấn luyện được của fold
    models.append(model)

    # Dự đoán trên tập validation của fold và tính các độ đo
    y_pred = np.argmax(model.predict(X_val_fold, verbose=0), axis=1)
    acc = metrics.accuracy_score(y_val_fold, y_pred)
    pre = metrics.precision_score(y_val_fold, y_pred, average='macro', zero_division=0)
    rec = metrics.recall_score(y_val_fold, y_pred, average='macro', zero_division=0)
    f1 = metrics.f1_score(y_val_fold, y_pred, average='macro', zero_division=0)
    accuracy_per_fold.append(acc)
    precision_per_fold.append(pre)
    recall_per_fold.append(rec)
    f1_per_fold.append(f1)

    p, r, f, _ = metrics.precision_recall_fscore_support(y_val_fold, y_pred, labels=[0, 1, 2], zero_division=0)
    class_precision.append(p)
    class_recall.append(r)
    class_f1.append(f)

    print(f'Fold {fold_no}: accuracy = {acc:.4f}; precision = {pre:.4f}; recall = {rec:.4f}; f1-score = {f1:.4f}')
    fold_no += 1

Fold 1: accuracy = 1.0000; precision = 1.0000; recall = 1.0000; f1-score = 1.0000
Fold 2: accuracy = 1.0000; precision = 1.0000; recall = 1.0000; f1-score = 1.0000
Fold 3: accuracy = 1.0000; precision = 1.0000; recall = 1.0000; f1-score = 1.0000
Fold 4: accuracy = 0.9333; precision = 0.9524; recall = 0.9333; f1-score = 0.9373
Fold 5: accuracy = 1.0000; precision = 1.0000; recall = 1.0000; f1-score = 1.0000
Fold 6: accuracy = 0.9333; precision = 0.9167; recall = 0.9444; f1-score = 0.9221
Fold 7: accuracy = 0.8667; precision = 0.9167; recall = 0.8333; f1-score = 0.8413
Fold 8: accuracy = 1.0000; precision = 1.0000; recall = 1.0000; f1-score = 1.0000
Fold 9: accuracy = 1.0000; precision = 1.0000; recall = 1.0000; f1-score = 1.0000
Fold 10: accuracy = 1.0000; precision = 1.0000; recall = 1.0000; f1-score = 1.0000


Đặng Phan Duy - 6451071010

In [7]:
# Kết quả của từng fold
fold_results = pd.DataFrame({
    'Fold': range(1, len(accuracy_per_fold) + 1),
    'Accuracy': accuracy_per_fold,
    'Precision': precision_per_fold,
    'Recall': recall_per_fold,
    'F1-score': f1_per_fold
})
fold_results

,Fold,Accuracy,Precision,Recall,F1-score
0,1,1.000000,1.000000,1.000000,1.000000
1,2,1.000000,1.000000,1.000000,1.000000
2,3,1.000000,1.000000,1.000000,1.000000
3,4,0.933333,0.952381,0.933333,0.937322
4,5,1.000000,1.000000,1.000000,1.000000
5,6,0.933333,0.916667,0.944444,0.922078
6,7,0.866667,0.916667,0.833333,0.841270
7,8,1.000000,1.000000,1.000000,1.000000
8,9,1.000000,1.000000,1.000000,1.000000
9,10,1.000000,1.000000,1.000000,1.000000


Đặng Phan Duy - 6451071010

## Câu e

In [8]:
# Average Accuracy, Recall, Precision, F1-score sau k-fold cross validation
print("Average Accuracy:", np.mean(accuracy_per_fold))
print("Average Recall:", np.mean(recall_per_fold))
print("Average Precision:", np.mean(precision_per_fold))
print("Average F1-score:", np.mean(f1_per_fold))

Average Accuracy: 0.9733333333333334
Average Recall: 0.9711111111111113
Average Precision: 0.9785714285714284
Average F1-score: 0.9700669700669702


Đặng Phan Duy - 6451071010

## Câu f

In [9]:
# Average Recall, Precision, F1-score cho từng class
class_results = pd.DataFrame({
    'Class': iris.target_names,
    'Average Recall': np.mean(class_recall, axis=0),
    'Average Precision': np.mean(class_precision, axis=0),
    'Average F1-score': np.mean(class_f1, axis=0)
})
class_results

,Class,Average Recall,Average Precision,Average F1-score
0,setosa,1.000000,1.000000,1.000000
1,versicolor,0.933333,0.985714,0.949883
2,virginica,0.980000,0.950000,0.960317


Đặng Phan Duy - 6451071010

## Câu g

In [10]:
# Chọn mô hình có F1-score tốt nhất giữa các fold
best_index = np.argmax(f1_per_fold)
best_model = models[best_index]
print("Best fold:", best_index + 1, "- F1-score:", f1_per_fold[best_index])

# 03 mẫu dữ liệu mới, chuẩn hóa giống như dữ liệu huấn luyện (chia cho max của từng cột)
X_new = np.array([[6.2, 2.9, 4.3, 1.3],
                  [5.1, 3.5, 1.4, 0.2],
                  [7.3, 2.8, 6.4, 2.1]])
X_new_scaled = X_new / X_max

new_prob = best_model.predict(X_new_scaled, verbose=0)
new_pred = np.argmax(new_prob, axis=1)
for sample, prob, label in zip(X_new, new_prob, new_pred):
    print(sample, "->", iris.target_names[label], "- Probability:", np.round(prob, 4))

Best fold: 1 - F1-score: 1.0
[6.2 2.9 4.3 1.3] -> versicolor - Probability: [0.000e+00 9.999e-01 1.000e-04]
[5.1 3.5 1.4 0.2] -> setosa - Probability: [1. 0. 0.]
[7.3 2.8 6.4 2.1] -> virginica - Probability: [0.000e+00 6.000e-04 9.994e-01]


Đặng Phan Duy - 6451071010